# S14 – Эмбеддинги, FAISS и поиск сходства по текстовым фрагментам

В этом ноутбуке разбираем **первый практический шаг к RAG**: как превратить набор текстов в векторы, как собрать индекс по этим векторам и как выполнять поиск **по смыслу**, а не только по точному совпадению слов.

Фокус именно на механике retrieval:

- готовим небольшой корпус документов;
- режем документы на чанки;
- строим векторные представления фрагментов;
- считаем сходство запроса и фрагментов;
- собираем индекс в `FAISS`;
- выполняем `top-k` поиск и интерпретируем результаты.

Для основной траектории ноутбука используется `sentence-transformers`. Если в локальной среде этот стек недоступен, ноутбук автоматически переключится на **TF-IDF fallback**, чтобы логика индексации и similarity search всё равно отработала. Это полезно, но важно понимать: **TF-IDF не заменяет плотные эмбеддинги**.

## 0. План

К концу ноутбука надо уметь:

1. Подготовить небольшой корпус документов для задач retrieval.
2. Разбить документы на текстовые чанки с overlap.
3. Получить векторные представления фрагментов текста.
4. Посчитать сходство между запросом и фрагментами вручную.
5. Собрать индекс `FAISS` и выполнять `top-k` поиск.
6. Интерпретировать результаты similarity search.
7. Видеть типичные ошибки: слишком общий запрос, плохой чанкинг, смешение тем внутри фрагмента.

## 1. Импорты и общие настройки

Ниже подключаем библиотеки, при необходимости устанавливаем недостающие зависимости, фиксируем `seed` и настраиваем воспроизводимое окружение.

В ноутбуке используются две логические группы инструментов:

- **векторизация текста** – через `sentence-transformers` (основной путь) или `TF-IDF` (fallback);
- **поиск ближайших соседей** – через `FAISS` (основной путь) или `NearestNeighbors` из `sklearn` (fallback).

In [1]:

# Базовые библиотеки для воспроизводимости, работы с данными и удобного вывода результатов.
import os
import sys
import random
import subprocess
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def ensure_package(package_name: str, import_name: Optional[str] = None) -> None:
    """Пытается импортировать пакет и при необходимости установить его через pip."""
    target = import_name or package_name
    try:
        __import__(target)
    except Exception:
        print(f"Устанавливаем пакет: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


# Для retrieval-контура попробуем установить основные зависимости.
# Даже если sentence-transformers не поднимется, ноутбук сможет работать через fallback.
ensure_package("faiss-cpu", "faiss")
ensure_package("sentence-transformers", "sentence_transformers")


try:
    import faiss  # type: ignore
    FAISS_AVAILABLE = True
except Exception as e:
    FAISS_AVAILABLE = False
    print("FAISS недоступен, будет использован fallback на sklearn NearestNeighbors.")
    print("Причина:", repr(e))


print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS available:", FAISS_AVAILABLE)


Устанавливаем пакет: faiss-cpu
NumPy: 2.0.2
Pandas: 2.2.2
FAISS available: True


In [2]:
# Фиксируем seed и определяем устройство.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)


set_seed(42)

try:
    import torch

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("Устройство для работы:", DEVICE)

Устройство для работы: cpu


## 2. Мини-корпус документов и постановка задачи

Чтобы не отвлекаться на внешние файлы, соберём **небольшую учебную базу знаний** прямо в ноутбуке. Все документы посвящены смежным темам: эмбеддинги, retrieval, RAG, FAISS, чанкинг и оценка качества извлечения.

Важно: в реальной задаче документы обычно длиннее и разнообразнее, а retrieval строится не по целым документам, а по их **фрагментам**. Поэтому уже на этом шаге полезно думать не в категориях «документ целиком», а в категориях «чанк как единица поиска».

In [3]:
# Небольшой учебный корпус документов по теме retrieval и RAG.
documents: List[Dict[str, str]] = [
    {
        "doc_id": "doc_01",
        "title": "Эмбеддинги текстов",
        "text": (
            "Эмбеддинг – это плотное векторное представление текста, в котором семантически похожие фразы "
            "располагаются близко друг к другу. Такие векторы позволяют искать похожие документы не по "
            "точному совпадению слов, а по смыслу запроса. Для задач retrieval эмбеддинги обычно "
            "нормализуют, чтобы косинусное сходство можно было считать через скалярное произведение."
        ),
    },
    {
        "doc_id": "doc_02",
        "title": "FAISS и быстрый поиск",
        "text": (
            "FAISS – библиотека для эффективного поиска ближайших соседей по векторам. "
            "В учебных примерах удобно начинать с точного индекса IndexFlatIP или IndexFlatL2, "
            "а затем переходить к более сложным структурам. Если эмбеддинги нормализованы, "
            "IndexFlatIP хорошо подходит для поиска по косинусному сходству."
        ),
    },
    {
        "doc_id": "doc_03",
        "title": "Чанкинг документов",
        "text": (
            "Перед индексацией длинные документы обычно режут на чанки – небольшие фрагменты текста. "
            "Слишком крупные чанки дают много лишнего контекста, а слишком мелкие могут терять смысл. "
            "На практике часто используют overlap, чтобы важная мысль не разрывалась на границе соседних фрагментов."
        ),
    },
    {
        "doc_id": "doc_04",
        "title": "Что такое RAG",
        "text": (
            "RAG объединяет два шага: сначала система извлекает релевантные фрагменты из базы знаний, "
            "а затем использует найденный контекст для генерации ответа. Качество RAG сильно зависит "
            "от retrieval: если нужный фрагмент не найден, генератор не сможет уверенно опереться на знания."
        ),
    },
    {
        "doc_id": "doc_05",
        "title": "Векторные базы данных",
        "text": (
            "Векторная база данных хранит эмбеддинги документов, метаданные и средства быстрого поиска "
            "по сходству. В учебных ноутбуках часто достаточно локального индекса в памяти, "
            "но в прикладных системах используют специализированные решения вроде pgvector, Milvus или Qdrant."
        ),
    },
    {
        "doc_id": "doc_06",
        "title": "Оценка retrieval",
        "text": (
            "Качество извлечения нельзя оценивать только по субъективному впечатлению от одного-двух запросов. "
            "Обычно задают набор контрольных вопросов и проверяют, попадает ли релевантный фрагмент в top-k. "
            "Для этого используют метрики hit@k, recall@k и анализ ошибок по неудачным запросам."
        ),
    },
    {
        "doc_id": "doc_07",
        "title": "Обновление базы знаний",
        "text": (
            "Если в базу знаний добавили новый документ, его недостаточно просто сохранить как текст. "
            "Нужно получить эмбеддинги новых фрагментов и обновить индекс поиска. "
            "Иначе retrieval продолжит работать по устаревшему состоянию знаний."
        ),
    },
    {
        "doc_id": "doc_08",
        "title": "Ограничения similarity search",
        "text": (
            "Similarity search хорошо находит близкие по смыслу фрагменты, но сам по себе не гарантирует истинную релевантность. "
            "Слишком общий запрос может вернуть много тематически похожих, но бесполезных отрывков. "
            "Поэтому retrieval всегда нужно проверять на конкретных сценариях использования."
        ),
    },
]

docs_df = pd.DataFrame(documents)
print("Размер корпуса:", len(docs_df))
display(docs_df[["doc_id", "title"]])

Размер корпуса: 8


,doc_id,title
0,doc_01,Эмбеддинги текстов
1,doc_02,FAISS и быстрый поиск
2,doc_03,Чанкинг документов
3,doc_04,Что такое RAG
4,doc_05,Векторные базы данных
5,doc_06,Оценка retrieval
6,doc_07,Обновление базы знаний
7,doc_08,Ограничения similarity search


## 3. Модель эмбеддингов и fallback-стратегия

Для плотных эмбеддингов удобно использовать готовую multilingual-модель из `sentence-transformers`. В ноутбуке ниже мы:

1. пытаемся поднять полноценный encoder;
2. если это не удалось, переключаемся на `TF-IDF fallback`;
3. в обоих случаях сохраняем единый интерфейс: `fit_documents(...)` и `encode_queries(...)`.

Такой подход полезен для учебной инженерной практики: одна и та же логика поиска работает поверх разных векторизаторов, а студент видит, где заканчивается **семантический retrieval** и начинается просто **лексическое сходство**.

In [4]:
# Единый интерфейс для двух вариантов векторизации: dense embeddings и fallback.
class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError


class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore

        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")


class TfidfFallbackBackend(EmbeddingBackend):
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
        self.backend_name = "TF-IDF fallback"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.fit_transform(texts).toarray()
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.transform(texts).toarray()
        return vectors.astype("float32")


def build_embedding_backend(
    model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device: str = "cpu",
) -> EmbeddingBackend:
    try:
        backend = SentenceTransformersBackend(model_name=model_name, device=device)
        print("Используем полноценные dense embeddings.")
        print("Бэкэнд:", backend.backend_name)
        return backend
    except Exception as e:
        print("Не удалось загрузить sentence-transformers encoder.")
        print("Причина:", repr(e))
        print("Переключаемся на TF-IDF fallback. Ноутбук останется рабочим,")
        print("но это уже не полноценные dense embeddings.")
        return TfidfFallbackBackend()


embedder = build_embedding_backend(device=DEVICE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Используем полноценные dense embeddings.
Бэкэнд: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


## 4. Чанкинг документов

В задачах retrieval редко индексируют длинный документ как один объект. Обычно документ разбивают на **чанки**:

- чтобы retrieval возвращал более точный и локальный контекст;
- чтобы в индексе не было слишком длинных и размытых фрагментов;
- чтобы потом было удобно отдавать найденные куски в RAG-конвейер.

Ниже используем простой чанкинг по словам. Это не идеальный вариант, но для учебной демонстрации он прозрачен и легко анализируется.

In [5]:
# Простая функция чанкинга по словам.
def chunk_text(text: str, chunk_size: int = 22, overlap: int = 5) -> List[str]:
    words = text.replace("\n", " ").split()

    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            continue

        chunks.append(" ".join(chunk_words))

        if start + chunk_size >= len(words):
            break

    return chunks


def build_chunks_dataframe(
    docs: List[Dict[str, str]],
    chunk_size: int = 22,
    overlap: int = 5,
) -> pd.DataFrame:
    rows = []

    for doc in docs:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_id, chunk in enumerate(chunks):
            rows.append(
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "chunk_id": chunk_id,
                    "chunk_text": chunk,
                    "n_words": len(chunk.split()),
                }
            )

    return pd.DataFrame(rows)


chunks_df = build_chunks_dataframe(documents, chunk_size=22, overlap=5)

print("Количество чанков:", len(chunks_df))
display(chunks_df.head(10))

Количество чанков: 17


,doc_id,title,chunk_id,chunk_text,n_words
0,doc_01,Эмбеддинги текстов,0,Эмбеддинг – это плотное векторное представлени...,22
1,doc_01,Эмбеддинги текстов,1,Такие векторы позволяют искать похожие докумен...,22
2,doc_01,Эмбеддинги текстов,2,"retrieval эмбеддинги обычно нормализуют, чтобы...",13
3,doc_02,FAISS и быстрый поиск,0,FAISS – библиотека для эффективного поиска бли...,22
4,doc_02,FAISS и быстрый поиск,1,"индекса IndexFlatIP или IndexFlatL2, а затем п...",22
5,doc_03,Чанкинг документов,0,Перед индексацией длинные документы обычно реж...,22
6,doc_03,Чанкинг документов,1,"лишнего контекста, а слишком мелкие могут теря...",22
7,doc_04,Что такое RAG,0,RAG объединяет два шага: сначала система извле...,22
8,doc_04,Что такое RAG,1,для генерации ответа. Качество RAG сильно зави...,21
9,doc_05,Векторные базы данных,0,Векторная база данных хранит эмбеддинги докуме...,22


## 5. Векторные представления чанков

Теперь превращаем каждый чанк в вектор. На этом шаге особенно важно следить за двумя вещами:

- **форма матрицы**: число строк должно совпадать с числом чанков;
- **нормировка**: для косинусного сходства удобно, когда длина каждого вектора близка к 1.

Если векторы нормализованы, similarity search можно выполнять через обычное скалярное произведение. Это и удобно, и быстро.

In [6]:

# Строим векторные представления для всех чанков.
chunk_texts = chunks_df["chunk_text"].tolist()
chunk_embeddings = embedder.fit_documents(chunk_texts)

print("Форма матрицы эмбеддингов:", chunk_embeddings.shape)

# Проверяем длины векторов.
# Если normalize_embeddings=True сработал корректно, все нормы должны быть ≈ 1.0.
# Это означает, что косинусное сходство далее можно считать через скалярное произведение.
vector_norms = np.linalg.norm(chunk_embeddings, axis=1)
print("Минимальная норма:", round(float(vector_norms.min()), 4))
print("Максимальная норма:", round(float(vector_norms.max()), 4))
print("Средняя норма:", round(float(vector_norms.mean()), 4))
print("→ Нормы ≈ 1.0: нормировка подтверждена, dot product = cosine similarity.")


Форма матрицы эмбеддингов: (17, 384)
Минимальная норма: 1.0
Максимальная норма: 1.0
Средняя норма: 1.0
→ Нормы ≈ 1.0: нормировка подтверждена, dot product = cosine similarity.


## 6. Ручной расчёт сходства: запрос против всех чанков

Прежде чем переходить к индексу, полезно один раз посчитать similarity search **вручную**. Тогда становится ясно, что retrieval по сути делает две вещи:

1. кодирует запрос в вектор;
2. ищет те векторы документов, которые ближе всего к вектору запроса.

Если эмбеддинги нормализованы, косинусное сходство между запросом и чанками можно считать через матричное умножение.

In [7]:
# Считаем сходство запроса и всех чанков без FAISS.
query = "Как искать похожие фрагменты текста по векторам с помощью FAISS?"
query_vector = embedder.encode_queries([query])

# Для нормализованных векторов это эквивалент косинусного сходства.
manual_scores = (chunk_embeddings @ query_vector.T).ravel()

top_k = 5
top_indices = np.argsort(manual_scores)[::-1][:top_k]

manual_results_df = chunks_df.iloc[top_indices].copy()
manual_results_df["score"] = manual_scores[top_indices]
manual_results_df["score"] = manual_results_df["score"].round(4)
manual_results_df.insert(0, "rank", range(1, len(manual_results_df) + 1))

display(Markdown(f"**Запрос:** {query}"))
display(manual_results_df[["rank", "doc_id", "title", "chunk_id", "score", "chunk_text"]])

**Запрос:** Как искать похожие фрагменты текста по векторам с помощью FAISS?

,rank,doc_id,title,chunk_id,score,chunk_text
1,1,doc_01,Эмбеддинги текстов,1,0.6728,Такие векторы позволяют искать похожие докумен...
2,2,doc_01,Эмбеддинги текстов,2,0.5207,"retrieval эмбеддинги обычно нормализуют, чтобы..."
3,3,doc_02,FAISS и быстрый поиск,0,0.4782,FAISS – библиотека для эффективного поиска бли...
15,4,doc_08,Ограничения similarity search,0,0.4690,Similarity search хорошо находит близкие по см...
9,5,doc_05,Векторные базы данных,0,0.4396,Векторная база данных хранит эмбеддинги докуме...


## 7. Индекс `FAISS` для быстрого поиска

Ручной перебор всех чанков полезен для понимания механики, но в реальных задачах так делать неэффективно. Поэтому обычно строят специальный индекс для поиска ближайших соседей.

В этом ноутбуке используем:

- `FAISS IndexFlatIP`, если `FAISS` доступен;
- `NearestNeighbors` из `sklearn`, если нужен fallback.

Снаружи интерфейс будет единым: `add(...)` и `search(...)`.

In [8]:
# Единая обёртка над FAISS и fallback-поиском.
class VectorSearchIndex:
    def __init__(self, dim: int) -> None:
        self.dim = dim
        self.backend_name = None
        self._faiss_index = None
        self._nn_index = None

        if FAISS_AVAILABLE:
            self._faiss_index = faiss.IndexFlatIP(dim)  # type: ignore[name-defined]
            self.backend_name = "FAISS IndexFlatIP"
        else:
            self._nn_index = NearestNeighbors(metric="cosine")
            self.backend_name = "sklearn NearestNeighbors fallback"

    def add(self, vectors: np.ndarray) -> None:
        vectors = vectors.astype("float32")

        if self._faiss_index is not None:
            self._faiss_index.add(vectors)
        else:
            self._nn_index.fit(vectors)

    def search(self, query_vectors: np.ndarray, top_k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")

        if self._faiss_index is not None:
            scores, indices = self._faiss_index.search(query_vectors, top_k)
            return scores, indices

        distances, indices = self._nn_index.kneighbors(query_vectors, n_neighbors=top_k)
        scores = 1.0 - distances
        return scores, indices


search_index = VectorSearchIndex(dim=chunk_embeddings.shape[1])
search_index.add(chunk_embeddings)

print("Индекс построен.")
print("Бэкэнд индекса:", search_index.backend_name)

Индекс построен.
Бэкэнд индекса: FAISS IndexFlatIP


In [9]:

# Удобная функция для поиска похожих чанков.
def search_similar_chunks(query: str, top_k: int = 5) -> pd.DataFrame:
    query_vectors = embedder.encode_queries([query])
    scores, indices = search_index.search(query_vectors, top_k=top_k)

    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk_row = chunks_df.iloc[int(idx)]
        rows.append(
            {
                "rank": rank,
                "doc_id": chunk_row["doc_id"],
                "title": chunk_row["title"],
                "chunk_id": int(chunk_row["chunk_id"]),
                "score": round(float(score), 4),
                "chunk_text": chunk_row["chunk_text"],
            }
        )

    return pd.DataFrame(rows)


# Проверяем FAISS-поиск на первом запросе.
# Результаты должны совпасть с ручным расчётом из секции 6:
# тот же top-k, те же фрагменты, те же оценки сходства.
faiss_query = "Почему для similarity search нужен индекс по векторам?"
faiss_results_df = search_similar_chunks(faiss_query, top_k=5)

display(Markdown(f"**Запрос:** {faiss_query}"))
display(faiss_results_df)


**Запрос:** Почему для similarity search нужен индекс по векторам?

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_08,Ограничения similarity search,0,0.7264,Similarity search хорошо находит близкие по см...
1,2,doc_01,Эмбеддинги текстов,1,0.6534,Такие векторы позволяют искать похожие докумен...
2,3,doc_01,Эмбеддинги текстов,2,0.6375,"retrieval эмбеддинги обычно нормализуют, чтобы..."
3,4,doc_02,FAISS и быстрый поиск,1,0.6170,"индекса IndexFlatIP или IndexFlatL2, а затем п..."
4,5,doc_02,FAISS и быстрый поиск,0,0.6054,FAISS – библиотека для эффективного поиска бли...


## 8. Пакетный запуск нескольких запросов

Хорошая практика – не смотреть только на один «удачный» запрос, а сразу прогонять несколько разных формулировок. Это позволяет увидеть:

- насколько retrieval устойчив к перефразированию;
- не путает ли он близкие темы;
- возвращает ли он действительно полезные фрагменты, а не просто похожие слова.

In [10]:
# Проверяем retrieval на нескольких запросах.
example_queries = [
    "Что такое эмбеддинги текстов и зачем они нужны?",
    "Как работает FAISS при поиске ближайших соседей?",
    "Зачем делить длинные документы на чанки?",
    "Почему качество RAG зависит от retrieval?",
    "Как понять, что поиск по базе знаний работает плохо?",
]

for current_query in example_queries:
    display(Markdown(f"### Запрос: `{current_query}`"))
    display(search_similar_chunks(current_query, top_k=3))

### Запрос: `Что такое эмбеддинги текстов и зачем они нужны?`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_01,Эмбеддинги текстов,0,0.5965,Эмбеддинг – это плотное векторное представлени...
1,2,doc_03,Чанкинг документов,0,0.5030,Перед индексацией длинные документы обычно реж...
2,3,doc_07,Обновление базы знаний,0,0.3753,"Если в базу знаний добавили новый документ, ег..."


### Запрос: `Как работает FAISS при поиске ближайших соседей?`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_02,FAISS и быстрый поиск,0,0.5127,FAISS – библиотека для эффективного поиска бли...
1,2,doc_08,Ограничения similarity search,0,0.2995,Similarity search хорошо находит близкие по см...
2,3,doc_05,Векторные базы данных,1,0.2528,"достаточно локального индекса в памяти, но в п..."


### Запрос: `Зачем делить длинные документы на чанки?`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_03,Чанкинг документов,0,0.7326,Перед индексацией длинные документы обычно реж...
1,2,doc_07,Обновление базы знаний,0,0.4681,"Если в базу знаний добавили новый документ, ег..."
2,3,doc_03,Чанкинг документов,1,0.4250,"лишнего контекста, а слишком мелкие могут теря..."


### Запрос: `Почему качество RAG зависит от retrieval?`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_04,Что такое RAG,1,0.7492,для генерации ответа. Качество RAG сильно зави...
1,2,doc_04,Что такое RAG,0,0.6419,RAG объединяет два шага: сначала система извле...
2,3,doc_07,Обновление базы знаний,1,0.5838,фрагментов и обновить индекс поиска. Иначе ret...


### Запрос: `Как понять, что поиск по базе знаний работает плохо?`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_07,Обновление базы знаний,1,0.5545,фрагментов и обновить индекс поиска. Иначе ret...
1,2,doc_04,Что такое RAG,1,0.5374,для генерации ответа. Качество RAG сильно зави...
2,3,doc_07,Обновление базы знаний,0,0.5315,"Если в базу знаний добавили новый документ, ег..."



## 9. Типичные ошибки и антипримеры

Similarity search легко производит убедительные результаты, но это не значит, что retrieval уже «решён». Ниже посмотрим на две частые проблемы.

**Нерелевантный по теме запрос.** Индекс всегда вернёт top-k – даже если запрос совершенно не связан с корпусом. Оценки сходства при этом будут заметно ниже, чем у релевантных запросов, а возвращённые фрагменты случайны.

**Расплывчатый, но тематически близкий запрос.** Когда запрос принадлежит той же области, но сформулирован слишком широко, модель может вернуть несколько фрагментов с похожими оценками, и выбрать «лучший» из них затруднительно.

Поэтому в инженерной практике полезно сравнивать:

- нерелевантный по теме запрос (ожидаем низкие оценки и случайные фрагменты);
- точный запрос с термином предметной области (ожидаем высокий score у конкретного фрагмента);
- уточнённый запрос через метрику оценки (ожидаем попадание в нужный document).


In [11]:

# Сравниваем слишком общий и более точный запросы.
# broad_query намеренно выбран нерелевантным теме корпуса:
# модель вернёт что-то, но это будет случайный шум из ближайших по эмбеддингу векторов.
broad_query = "сегодня хорошая погода"
specific_query = "как FAISS ищет ближайших соседей по векторам"
domain_query = "почему retrieval оценивают через hit@k и recall@k"

for current_query in [broad_query, specific_query, domain_query]:
    display(Markdown(f"### Запрос: `{current_query}`"))
    display(search_similar_chunks(current_query, top_k=4))


### Запрос: `сегодня хорошая погода`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_02,FAISS и быстрый поиск,0,0.0847,FAISS – библиотека для эффективного поиска бли...
1,2,doc_02,FAISS и быстрый поиск,1,0.0427,"индекса IndexFlatIP или IndexFlatL2, а затем п..."
2,3,doc_05,Векторные базы данных,0,0.0405,Векторная база данных хранит эмбеддинги докуме...
3,4,doc_01,Эмбеддинги текстов,2,0.0124,"retrieval эмбеддинги обычно нормализуют, чтобы..."


### Запрос: `как FAISS ищет ближайших соседей по векторам`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_02,FAISS и быстрый поиск,0,0.6433,FAISS – библиотека для эффективного поиска бли...
1,2,doc_01,Эмбеддинги текстов,1,0.4264,Такие векторы позволяют искать похожие докумен...
2,3,doc_05,Векторные базы данных,1,0.4242,"достаточно локального индекса в памяти, но в п..."
3,4,doc_01,Эмбеддинги текстов,2,0.4029,"retrieval эмбеддинги обычно нормализуют, чтобы..."


### Запрос: `почему retrieval оценивают через hit@k и recall@k`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_06,Оценка retrieval,1,0.7321,"проверяют, попадает ли релевантный фрагмент в ..."
1,2,doc_07,Обновление базы знаний,1,0.5433,фрагментов и обновить индекс поиска. Иначе ret...
2,3,doc_08,Ограничения similarity search,1,0.5149,общий запрос может вернуть много тематически п...
3,4,doc_01,Эмбеддинги текстов,2,0.5102,"retrieval эмбеддинги обычно нормализуют, чтобы..."


## 10. Итоги

1. **Retrieval начинается не с LLM, а с векторизации и индекса.** Без качественного retrieval полноценный RAG не работает.
2. **Чанкинг влияет на качество поиска.** Слишком длинные и слишком короткие фрагменты одинаково вредны.
3. **Эмбеддинги позволяют искать по смыслу, а не только по совпадению слов.**
4. **FAISS даёт удобный и быстрый индекс для top-k поиска по векторам.**
5. **Similarity search надо проверять на наборе запросов, а не на одном удачном примере.**
6. **Даже хорошие результаты retrieval нужно интерпретировать критически:** похожесть ещё не равна реальной полезности ответа.

## Задания для самостоятельной работы

1. Добавьте в учебный корпус 3-5 собственных документов и перестройте индекс.
2. Попробуйте изменить `chunk_size` и `overlap`, затем сравните качество retrieval на 5 запросах.
3. Составьте маленький набор контрольных вопросов и проверьте, попадает ли нужный фрагмент в `top-3`.
4. Добавьте поле `source` или `section` в метаданные чанков и выводите его вместе с результатами поиска.
5. Попробуйте заменить учебный корпус на собственный набор заметок, регламентов или конспектов и проверьте, насколько retrieval устойчив к реальным формулировкам запросов.